# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is. The functions below
are the ones behind the Monitoring page of the app.

To make sure there is something to show, this notebook first runs all six demo
scenarios in mock mode (approving only A and B, as the CLI demo does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

A completed


B completed
C blocked
D blocked
E blocked


F completed


In [3]:
from quantlab.monitoring import (
    capability_counts,
    guardrail_counts,
    kpis,
    read_events,
    recent_errors,
    stage_latency,
)

events = read_events(settings.events_path)
print(len(events), "events")
pd.Series(kpis(events))

303 events


n_runs               24.0000
n_completed          14.0000
n_blocked            10.0000
n_failed              0.0000
n_in_progress         0.0000
success_rate          0.5833
blocked_rate          0.4167
failed_rate           0.0000
n_replays             3.0000
replay_match_rate     1.0000
cache_hit_rate           NaN
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds):

In [4]:
pd.DataFrame(stage_latency(events))

,stage,count,mean_ms,p95_ms
0,planning,24,0.0,0.0
1,validation,24,0.3,0.8
2,execution,14,306.8,448.5
3,narration,14,0.0,0.0


Which capabilities run most, and which guardrails fire most:

In [5]:
pd.Series(capability_counts(events), name="executions")

load_series           20
resample              20
compute_returns        8
rolling_volatility     8
flag_regime            8
forward_return         8
compare_groups         8
ols_hac                8
spread                 6
summary_stats          6
difference             6
adf_test               6
Name: executions, dtype: int64

In [6]:
pd.DataFrame(guardrail_counts(events))

,code,name,severity,count
0,G12,MATERIAL_ASSUMPTION,APPROVAL,15
1,R04,ALIGNMENT_LOSS,WARN,6
2,G10,LOOK_AHEAD,APPROVAL,3
3,G11,OVERLAPPING_OBSERVATIONS,WARN,3
4,G08,RETURNS_ON_RATES,BLOCK,3
5,G02,UNKNOWN_CAPABILITY,BLOCK,3
6,G13,PLANNER_REFUSED,BLOCK,3
7,G09,FREQUENCY_MISMATCH,BLOCK,3


The most recent failures (blocked runs and errors):

In [7]:
pd.DataFrame(recent_errors(events, limit=5))

,timestamp,run_id,event_type,error
0,2026-10-14T20:12:18.553Z,20261014-201218-8de527,run_failed,The plan has blocking issues: G09:s7
1,2026-10-14T20:12:18.494Z,20261014-201218-9e1e04,run_failed,The plan has blocking issues: G13:plan
2,2026-10-14T20:12:18.423Z,20261014-201218-6ecb12,run_failed,"The plan has blocking issues: G08:s3, G02:s4"
3,2026-10-14T20:11:06.730Z,20261014-201106-edcc28,run_failed,The plan has blocking issues: G09:s7
4,2026-10-14T20:11:06.652Z,20261014-201106-2280df,run_failed,The plan has blocking issues: G13:plan
